# ViT-Up Colab Inference Example

Minimal Google Colab notebook: clone the repo, install inference dependencies, load a pretrained ViT-Up wrapper, run dense queries on one image, and visualize the features with PCA.

In Colab, select **Runtime > Change runtime type > GPU** before running the notebook.

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo_dir = Path("/content/vit-up")
    if repo_dir.exists():
        subprocess.run(["git", "-C", str(repo_dir), "fetch", "origin", "main"], check=True)
        subprocess.run(["git", "-C", str(repo_dir), "reset", "--hard", "origin/main"], check=True)
    else:
        subprocess.run(
            ["git", "clone", "https://github.com/krispinwandel/vit-up.git", str(repo_dir)],
            check=True,
        )
    os.chdir(repo_dir)
    subprocess.run(
        [
            sys.executable,
            "-m",
            "pip",
            "install",
            "-q",
            "huggingface_hub",
            "safetensors",
            "omegaconf",
            "peft",
            "transformers",
            "timm",
            "einops",
            "matplotlib",
            "pillow",
            "scikit-learn",
            "torchao>=0.16.0",
        ],
        check=True,
    )
else:
    repo_dir = Path.cwd().resolve()
    if repo_dir.name == "notebooks":
        repo_dir = repo_dir.parent
    os.chdir(repo_dir)

if str(Path.cwd()) not in sys.path:
    sys.path.insert(0, str(Path.cwd()))

for module_name in list(sys.modules):
    if module_name == "vit_up" or module_name.startswith("vit_up."):
        del sys.modules[module_name]

print(f"Repository: {Path.cwd()}")

DINOv3 is gated on Hugging Face. Before loading the model, make sure your Hugging Face account has access to the DINOv3 model page, then log in below with a token that can read gated repositories.

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

In [ ]:
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import torchvision.transforms as T
from PIL import Image

from vit_up.inference.vit_up_wrapper import ViTUpWrapper
from vit_up.utils import pca_utils

device = "cuda" if torch.cuda.is_available() else "cpu"
use_bfloat16 = bool(device == "cuda" and torch.cuda.is_bf16_supported())
torch.set_grad_enabled(False)

print(f"PyTorch: {torch.__version__}")
print(f"Device: {device}")
print(f"bf16: {use_bfloat16}")

In [ ]:
model_name = "vit_up_dinov3_splus"  # or "vit_up_dinov3_base"
query_res = 448
query_chunk_size = 4096

model = ViTUpWrapper(
    model_name,
    device=device,
    use_bfloat16=use_bfloat16,
    query_chunk_size=query_chunk_size,
).eval()

In [ ]:
image_path = Path("assets") / "fruit_store.png"
image = Image.open(image_path).convert("RGB")
image = T.CenterCrop(min(image.size))(image)

model.set_images(image)
cache_data = model.get_cache_data()
last_hidden_states = cache_data["layer_hidden_states_hwc"][-1][0].detach().float().cpu()
low_h, low_w, feat_dim = last_hidden_states.shape

low_features = last_hidden_states.reshape(low_h * low_w, feat_dim)
low_pca_scores, pca_mean, pca_std, _, pca_eigenvectors = pca_utils.pca(
    low_features,
    k=3,
    std_normalize=True,
)
low_pca_rgb, rgb_min, rgb_max = pca_utils.tensor_to_rgb(low_pca_scores)
low_pca_rgb = low_pca_rgb.reshape(low_h, low_w, 3)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(image)
axes[0].set_title("Input")
axes[0].axis("off")

axes[1].imshow(low_pca_rgb.numpy())
axes[1].set_title(f"Cached hidden-state PCA ({low_h}x{low_w})")
axes[1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
coords = (torch.arange(query_res, device=device, dtype=torch.float32) + 0.5) / query_res
grid_y, grid_x = torch.meshgrid(coords, coords, indexing="ij")
query_coords = torch.stack((grid_x, grid_y), dim=-1).reshape(1, query_res * query_res, 2)

features = model(query_coords=query_coords)
features = features[0].detach().float().cpu()

print(f"Features: {tuple(features.shape)}")

In [ ]:
high_pca_scores = pca_utils.apply_pca(features, pca_eigenvectors, pca_mean, pca_std)
high_pca_rgb, _, _ = pca_utils.tensor_to_rgb(
    high_pca_scores,
    tensor_min=rgb_min,
    tensor_max=rgb_max,
)
high_pca_rgb = high_pca_rgb.reshape(query_res, query_res, 3)

low_pca_rgb_upsampled = F.interpolate(
    low_pca_rgb.permute(2, 0, 1).unsqueeze(0).float(),
    size=(query_res, query_res),
    mode="nearest",
)[0].permute(1, 2, 0).byte()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
axes[0].imshow(image)
axes[0].set_title("Input")
axes[0].axis("off")

axes[1].imshow(low_pca_rgb_upsampled.numpy())
axes[1].set_title(f"Cached PCA NN upsampled ({query_res}x{query_res})")
axes[1].axis("off")

axes[2].imshow(high_pca_rgb.numpy())
axes[2].set_title(f"ViT-Up PCA ({query_res}x{query_res})")
axes[2].axis("off")

plt.tight_layout()
plt.show()